# Capstone Week9: Parquet Data Processing

This notebook processes `capstone_project_1000_data.parquet` step-by-step:
1) Remove logical exceptions in dates (treat `ref_branch_end_date==1900-01-01` as open-ended).
2) Clamp negative `RSV` to 0.
3) Deduplicate by (`ref_branch_code`, `material_nature_sum_desc`, `stock_out_date`).
4) Monthly aggregation (sum `RSV` and average `price`) per (`ref_branch_code`, `material_nature_sum_desc`), outputting month start as `stock_out_date`.
5) Check missing months within the min/max date range for each (`ref_branch_code`, `material_nature_sum_desc`).
6) Fill missing months' `RSV` and `price` using previous 3-month moving average and same-month last-year value.


In [18]:
import pandas as pd
import numpy as np

INPUT_PATH = "capstone_project_1000_data.parquet"

# Output files (optional)
SAVE_OUTPUT = True
OUT_MONTHLY_PARQUET = "capstone_monthly_agg.parquet"
OUT_MONTHLY_FILLED_PARQUET = "monthly_aggregated_filled.parquet"
OUT_MISSING_SUMMARY_CSV = "capstone_missing_months_summary.csv"
OUT_MISSING_DETAIL_CSV = "capstone_missing_months_detail.csv"


In [19]:
# Step 0: Load only the required columns
cols = [
    "ref_branch_code",
    "material_nature_sum_desc",
    "stock_out_date",
    "RSV",
    "price",
    "ref_branch_start_date",
    "ref_branch_end_date",
]

df = pd.read_parquet(INPUT_PATH, columns=cols, engine="pyarrow")

# Ensure datetime types
for c in ["stock_out_date", "ref_branch_start_date", "ref_branch_end_date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

print("Loaded rows:", len(df))
print("Date range (stock_out_date):", df["stock_out_date"].min(), "~", df["stock_out_date"].max())
print("Columns:", list(df.columns))


Loaded rows: 4432053
Date range (stock_out_date): 2017-01-01 00:00:00 ~ 2025-12-31 00:00:00
Columns: ['ref_branch_code', 'material_nature_sum_desc', 'stock_out_date', 'RSV', 'price', 'ref_branch_start_date', 'ref_branch_end_date']


In [20]:
# Step 1: Remove logical exception rows per-row range check.
# Note: `ref_branch_end_date == 1900-01-01` is treated as +infinity (open-ended),
# so the upper bound constraint is skipped for those rows.
infinite_end = pd.Timestamp("1900-01-01")
invalid = (
    (df["stock_out_date"] < df["ref_branch_start_date"]) |
    ((df["ref_branch_end_date"] != infinite_end) & (df["stock_out_date"] > df["ref_branch_end_date"]))
)
mask_valid = ~invalid

df1 = df.loc[mask_valid].copy()
print("After step 1 (invalid date rows removed):", len(df1))
print("Step 1 deleted rows:", len(df) - len(df1))

# Step 2: RSV < 0 -> 0
df1["RSV"] = df1["RSV"].clip(lower=0)

# Keep only columns needed for step 3/4
df1 = df1[[
    "ref_branch_code",
    "material_nature_sum_desc",
    "stock_out_date",
    "RSV",
    "price",
]]
print("After step 2: negative RSV clamped")


After step 1 (invalid date rows removed): 4432017
Step 1 deleted rows: 36
After step 2: negative RSV clamped


In [21]:
# Step 3: Deduplicate by 3 keys.
# RSV -> mean; price -> first (deterministic by sorting).
keys3 = ["ref_branch_code", "material_nature_sum_desc", "stock_out_date"]

df1 = df1.sort_values(keys3)

df3 = (
    df1.groupby(keys3, as_index=False)
        .agg(
            RSV=("RSV", "mean"),
            price=("price", "first"),
        )
)

print("After step 3 (deduplicated):", len(df3))
print(df3.head())


After step 3 (deduplicated): 4430982
  ref_branch_code material_nature_sum_desc stock_out_date         RSV   price
0         CAT_008                  CAT_001     2023-01-01   4841.6510  453.90
1         CAT_008                  CAT_001     2023-01-02   3362.6000  452.20
2         CAT_008                  CAT_001     2023-01-03   6958.9500  452.20
3         CAT_008                  CAT_001     2023-01-06  15141.4325  453.05
4         CAT_008                  CAT_001     2023-01-07  19085.7810  456.45


In [22]:
# Step 4: Monthly aggregation per (`ref_branch_code`, `material_nature_sum_desc`).
# stock_out_date -> month start (1st day)
# RSV -> sum; price -> mean
df3 = df3.copy()
df3["stock_out_month"] = df3["stock_out_date"].dt.to_period("M").dt.to_timestamp()

df4 = (
    df3.groupby(["ref_branch_code", "material_nature_sum_desc", "stock_out_month"], as_index=False)
       .agg(
           RSV=("RSV", "sum"),
           price=("price", "mean"),
       )
)

# Rename to match your requested output field names
df4 = df4.rename(columns={
    "material_nature_sum_desc": "material_nature_sum",
    "stock_out_month": "stock_out_date",
})

# Keep only requested fields
df4 = df4[["ref_branch_code", "material_nature_sum", "stock_out_date", "RSV", "price"]]

print("After step 4 (monthly agg):", len(df4))
print(df4.head())


After step 4 (monthly agg): 438467
  ref_branch_code material_nature_sum stock_out_date         RSV       price
0         CAT_008             CAT_001     2023-01-01  256468.426  460.994231
1         CAT_008             CAT_001     2023-02-01  230430.495  455.273077
2         CAT_008             CAT_001     2023-03-01  114700.734  470.900000
3         CAT_008             CAT_001     2023-04-01  188948.217  491.029545
4         CAT_008             CAT_001     2023-05-01  253250.105  495.918333


In [23]:
# Step 5: Missing months check for each (`ref_branch_code`, `material_nature_sum`).
keys5 = ["ref_branch_code", "material_nature_sum"]

df4_sorted = df4.sort_values(keys5 + ["stock_out_date"]).copy()

summary_rows = []
detail_rows = []

for (bcode, msum), g in df4_sorted.groupby(keys5, sort=False):
    min_dt = g["stock_out_date"].min()
    max_dt = g["stock_out_date"].max()

    expected = pd.date_range(min_dt, max_dt, freq="MS")  # month start
    present = pd.DatetimeIndex(g["stock_out_date"].dropna().unique())
    missing = expected.difference(present)

    total_expected = len(expected)
    missing_count = len(missing)
    missing_rate = (missing_count / total_expected) if total_expected else np.nan

    summary_rows.append({
        "ref_branch_code": bcode,
        "material_nature_sum": msum,
        "min_stock_out_date": min_dt,
        "max_stock_out_date": max_dt,
        "expected_months": total_expected,
        "present_months": len(present),
        "missing_months_count": missing_count,
        "missing_rate": missing_rate,
    })

    for dt in missing:
        detail_rows.append({
            "ref_branch_code": bcode,
            "material_nature_sum": msum,
            "missing_stock_out_date": dt,
        })

missing_summary = pd.DataFrame(summary_rows)
missing_detail = pd.DataFrame(detail_rows)

print("Missing summary rows:", len(missing_summary))
print("Missing detail rows:", len(missing_detail))
if len(missing_summary) > 0:
    display(missing_summary.sort_values("missing_rate", ascending=False).head(10))


Missing summary rows: 9507
Missing detail rows: 70441


,ref_branch_code,material_nature_sum,min_stock_out_date,max_stock_out_date,expected_months,present_months,missing_months_count,missing_rate
8550,CAT_8042,CAT_006,2019-06-01,2025-05-01,72,2,70,0.972222
8890,CAT_8312,CAT_006,2017-01-01,2025-02-01,98,3,95,0.969388
1877,CAT_2342,CAT_006,2021-03-01,2025-11-01,57,2,55,0.964912
8062,CAT_7697,CAT_006,2021-12-01,2025-12-01,49,2,47,0.959184
6040,CAT_6067,CAT_006,2021-12-01,2025-08-01,45,2,43,0.955556
1626,CAT_2154,CAT_006,2020-05-01,2025-11-01,67,3,64,0.955224
1985,CAT_2449,CAT_006,2020-04-01,2023-11-01,44,2,42,0.954545
3109,CAT_3460,CAT_006,2018-07-01,2025-10-01,88,4,84,0.954545
2090,CAT_2525,CAT_006,2021-05-01,2024-11-01,43,2,41,0.953488
4193,CAT_4460,CAT_006,2022-05-01,2025-11-01,43,2,41,0.953488


In [24]:
# Step 5.1: Overall missing-rate distribution (by (`ref_branch_code`, `material_nature_sum_desc`) pair)
# missing_rate is calculated within each pair's min/max monthly range.
rate = missing_summary["missing_rate"].fillna(0)
total_pairs = len(rate)

# Bins: <=20%, 20-40%, 40-60%, 60-80%, >=80%
c_20_or_less = int((rate <= 0.2).sum())
c_20_40 = int(((rate > 0.2) & (rate <= 0.4)).sum())
c_40_60 = int(((rate > 0.4) & (rate <= 0.6)).sum())
c_60_80 = int(((rate > 0.6) & (rate < 0.8)).sum())
c_80_plus = int((rate >= 0.8).sum())

print("Total pairs checked:", total_pairs)
print("Missing rate >= 80%:", c_80_plus)
print("Missing rate 60-80%:", c_60_80)
print("Missing rate 40-60%:", c_40_60)
print("Missing rate 20-40%:", c_20_40)
print("Missing rate <= 20%:", c_20_or_less)

# Optional: sanity check that bins cover all rows
bin_sum = c_80_plus + c_60_80 + c_40_60 + c_20_40 + c_20_or_less
print("Sum of bin counts:", bin_sum)
if total_pairs != bin_sum:
    print("[Warning] Bin counts do not match total; check boundary handling.")

# Drop pairs with missing_rate >= 80% before filling
keep_pairs = missing_summary.loc[missing_summary["missing_rate"].fillna(0) < 0.8, [
    "ref_branch_code", "material_nature_sum"
]]
drop_pairs_count = int((missing_summary["missing_rate"].fillna(0) >= 0.8).sum())
print("Pairs to drop (missing_rate >= 80%):", drop_pairs_count)

before_rows = len(df4)
df4 = df4.merge(keep_pairs, on=["ref_branch_code", "material_nature_sum"], how="inner")
after_rows = len(df4)
print("df4 rows after dropping:", after_rows, "(from", before_rows, ")")
print("Pairs kept:", len(keep_pairs))


Total pairs checked: 9507
Missing rate >= 80%: 219
Missing rate 60-80%: 444
Missing rate 40-60%: 745
Missing rate 20-40%: 1355
Missing rate <= 20%: 6744
Sum of bin counts: 9507
Pairs to drop (missing_rate >= 80%): 219
df4 rows after dropping: 437260 (from 438467 )
Pairs kept: 9288


In [25]:
# Step 6: Fill missing months to form a complete monthly time series
# For each (`ref_branch_code`, `material_nature_sum`):
# - Create full month index between min and max.
# - If RSV/price is missing, fill with the average of:
#   a) previous 3-month moving average (we iterate so consecutive missing months can use imputed values)
#   b) same month last year (t-12)

keys6 = ["ref_branch_code", "material_nature_sum"]
df4_sorted = df4.sort_values(keys6 + ["stock_out_date"]).copy()

filled_parts = []

for (bcode, msum), g in df4_sorted.groupby(keys6, sort=False):
    g = g.sort_values("stock_out_date")
    g = g.set_index("stock_out_date")

    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="MS")
    g_full = g.reindex(full_idx)

    # restore key columns
    g_full["ref_branch_code"] = bcode
    g_full["material_nature_sum"] = msum

    # Iteratively fill so consecutive missing months can use newly imputed values.
    rsv = g_full["RSV"].copy()
    price = g_full["price"].copy()
    for i in range(len(rsv)):
        if pd.isna(rsv.iat[i]):
            mov3 = rsv.iloc[max(0, i - 3):i].mean()  # up to previous 3 months
            yoy = rsv.iat[i - 12] if i >= 12 else np.nan
            rsv.iat[i] = np.nanmean([mov3, yoy])

        if pd.isna(price.iat[i]):
            mov3 = price.iloc[max(0, i - 3):i].mean()  # up to previous 3 months
            yoy = price.iat[i - 12] if i >= 12 else np.nan
            price.iat[i] = np.nanmean([mov3, yoy])

    g_full["RSV"] = rsv
    g_full["price"] = price

    g_full = g_full.reset_index().rename(columns={"index": "stock_out_date"})
    filled_parts.append(g_full[["ref_branch_code", "material_nature_sum", "stock_out_date", "RSV", "price"]])

df_full = pd.concat(filled_parts, ignore_index=True)
df_full = df_full.sort_values(keys6 + ["stock_out_date"]).reset_index(drop=True)

# Final column rename to match downstream requirement
df_final = df_full.rename(columns={
    "stock_out_date": "month",
    "RSV": "monthly_sales",
    "material_nature_sum": "material_nature_sum_desc",
})

# Keep only required columns
df_final = df_final[["ref_branch_code", "material_nature_sum_desc", "month", "monthly_sales", "price"]]

print("After step 6 (filled monthly series):", len(df_final))
print("monthly_sales missing after fill:", df_final["monthly_sales"].isna().sum())
print("price missing after fill:", df_final["price"].isna().sum())
print(df_final.head())


After step 6 (filled monthly series): 499318
monthly_sales missing after fill: 0
price missing after fill: 0
  ref_branch_code material_nature_sum_desc      month  monthly_sales  \
0         CAT_008                  CAT_001 2023-01-01     256468.426   
1         CAT_008                  CAT_001 2023-02-01     230430.495   
2         CAT_008                  CAT_001 2023-03-01     114700.734   
3         CAT_008                  CAT_001 2023-04-01     188948.217   
4         CAT_008                  CAT_001 2023-05-01     253250.105   

        price  
0  460.994231  
1  455.273077  
2  470.900000  
3  491.029545  
4  495.918333  


In [26]:
# Optional: Save results
if SAVE_OUTPUT:
    df4.to_parquet(OUT_MONTHLY_PARQUET, index=False)
    df_final.to_parquet(OUT_MONTHLY_FILLED_PARQUET, index=False)
    missing_summary.to_csv(OUT_MISSING_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    missing_detail.to_csv(OUT_MISSING_DETAIL_CSV, index=False, encoding="utf-8-sig")
    print("Saved:")
    print(" -", OUT_MONTHLY_PARQUET)
    print(" -", OUT_MONTHLY_FILLED_PARQUET)
    print(" -", OUT_MISSING_SUMMARY_CSV)
    print(" -", OUT_MISSING_DETAIL_CSV)


Saved:
 - capstone_monthly_agg.parquet
 - monthly_aggregated_filled.parquet
 - capstone_missing_months_summary.csv
 - capstone_missing_months_detail.csv
